# Proyek Data Science  Pengganti UTS
## Analisis Klastering Pola Konsumsi Energi Rumah Tangga
### untuk Efisiensi Distribusi PLN

---

| Informasi | Detail |
|-----------|--------|
| Nama Kelompok | Kelompok 3 |
| Anggota 1 | 301240040, Tegar Bagus Permana, Project Lead |
| Anggota 2 | 301240037, Sigit Miraj Permana, ML Engineer |
| Anggota 3 | 301240041, Selsa Shafana Alifiyani, Data Analyst |
| Anggota 4 | 301240030, Sony Moch Leviansyah, Data Engineer |
| Mata Kuliah | Data Science |
| Topik | Klastering Pola Konsumsi Energi Rumah Tangga untuk Efisiensi Distribusi PLN |
| Dataset | UCI Machine Learning Repository |
| Tanggal | Mei 2026 |

---

## Setup Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import urllib.request
import zipfile
import os
import missingno as msno 

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

---
# Fase 1  Business Understanding

## 1.1 Latar Belakang

PLN menghadapi masalah fluktuasi beban listrik yang signifikan antara jam sibuk dan jam lengang. 
Ketika jutaan rumah tangga menyalakan perangkat berdaya tinggi secara bersamaan pada malam hari, 
jaringan distribusi mengalami tekanan yang dapat mempercepat degradasi trafo dan memicu pemadaman lokal.

Pendekatan konvensional PLN selama ini berfokus pada penambahan kapasitas fisik (supply-side management), 
seperti membangun trafo atau pembangkit baru. Strategi ini membutuhkan biaya modal besar dan tidak efisien 
jika kapasitas tambahan hanya terpakai beberapa jam per hari. Pendekatan berbasis data memungkinkan PLN 
memahami pola konsumsi pelanggan secara granular tanpa harus mengandalkan ekspansi infrastruktur.

## 1.2 Rumusan Masalah

1. Bagaimana karakteristik pola konsumsi energi harian rumah tangga berdasarkan data pengukuran per menit?
2. Berapa jumlah segmen pelanggan yang optimal untuk merepresentasikan variasi konsumsi tanpa kehilangan interpretabilitas?
3. Fitur kelistrikan mana yang paling membedakan perilaku antar segmen, dan bagaimana hasilnya dapat dimanfaatkan PLN?

## 1.3 Tujuan Analitik

- Membangun model K-Means Clustering untuk mengelompokkan pola konsumsi energi rumah tangga.
- Mengidentifikasi karakteristik tiap segmen berdasarkan daya aktif, daya reaktif, dan sub-metering.
- Memetakan kurva beban harian per klaster untuk mendeteksi jam puncak yang membebani distribusi PLN.

Hasil analisis diharapkan menjadi dasar empiris bagi PLN dalam merancang kebijakan tarif dinamis 
dan menentukan prioritas pemeliharaan gardu distribusi.

## 1.4 Justifikasi Pendekatan Data-Driven

Pendekatan konvensional PLN dalam mengelola beban distribusi selama ini bersifat reaktif, yaitu menambah kapasitas infrastruktur ketika jaringan sudah mendekati batas maksimum. Metode ini tidak efisien karena kapasitas yang dibangun hanya terpakai dalam rentang jam sibuk yang singkat, sementara pada jam lengang kapasitas tersebut menganggur.

Pendekatan berbasis data memungkinkan PLN memahami perilaku konsumsi pelanggan secara granular tanpa harus mengandalkan ekspansi fisik. Dengan mengelompokkan pelanggan ke dalam segmen berdasarkan pola konsumsi aktual, PLN dapat:

1. Merancang kebijakan tarif yang lebih tepat sasaran berdasarkan profil konsumsi nyata, bukan asumsi umum.
2. Menentukan prioritas pemeliharaan gardu distribusi berdasarkan kepadatan klaster beban puncak di suatu zona.
3. Mengukur dampak program demand response secara kuantitatif menggunakan pergerakan anggota antarklaster.



# Fase 2  Data Understanding

## 2.1 Sumber dan Pengumpulan Data

Dataset yang digunakan adalah Individual Household Electric Power Consumption dari UCI Machine Learning Repository.
Data mencatat pengukuran konsumsi listrik satu rumah tangga di Prancis dari Desember 2006 hingga November 2010 
dengan granularitas per menit. Pengumpulan dilakukan melalui unduhan terprogram langsung dari server UCI.

Untuk efisiensi komputasi, analisis dibatasi pada 100.000 observasi pertama.

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip'

if not os.path.exists('household_power_consumption.txt'):
    urllib.request.urlretrieve(url, 'power.zip')
    with zipfile.ZipFile('power.zip', 'r') as zip_ref:
        zip_ref.extractall()

df = pd.read_csv('household_power_consumption.txt', sep=';',
                 na_values=['?'],
                 low_memory=False,
                 nrows=100000)

print(f'Dataset dimuat: {df.shape[0]} baris, {df.shape[1]} kolom')
df.head()

## 2.2 Eksplorasi Awal

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

## 2.3 Ringkasan Temuan Data Understanding

- **Jumlah observasi:** 100.000 baris, 9 kolom (2 waktu + 7 numerik kelistrikan)
- **Variabel target (Y):** Tidak ada — ini merupakan masalah unsupervised learning (klastering)
- **Masalah kualitas yang ditemukan:**
  - Terdapat missing values akibat karakter `?` pada file sumber yang terbaca sebagai NaN, 
    tersebar di 7 kolom numerik dengan proporsi kurang dari 0,5% dari total data
  - Kolom Date dan Time masih terpisah, belum bisa digunakan sebagai indeks waktu
  - Semua kolom numerik bertipe object karena keberadaan karakter `?`, perlu konversi eksplisit
- **Rencana penanganan:**
  - Missing values → listwise deletion karena proporsinya sangat kecil (< 0,5%)
  - Kolom Date + Time → digabung menjadi indeks datetime
  - Tipe data → konversi ke numerik dengan `pd.to_numeric(errors='coerce')`
  - Fitur turunan → tambah `Hour` dan `Is_Weekend` dari indeks datetime

---
# Fase 3  Data Preprocessing

## 3.0 Salin Dataset

Sebelum memulai preprocessing, dataset asli disalin ke variabel baru agar data mentah tetap tersedia dan tidak termodifikasi selama proses pembersihan berlangsung.


In [ ]:
df_clean = df.copy()
print(f'Dataset berhasil disalin: {df_clean.shape[0]:,} baris, {df_clean.shape[1]} kolom')
df_clean.head(3)


## 3.1 Penanganan Missing Values

Nilai hilang pada dataset ini muncul karena karakter tanda tanya (`?`) pada file sumber terbaca sebagai `NaN` saat proses loading. Visualisasi dilakukan terlebih dahulu untuk melihat pola sebaran missing values sebelum menentukan strategi penanganan.

Karena proporsi nilai hilang kurang dari 0,5% dari total data, penanganan dilakukan dengan listwise deletion (menghapus baris yang mengandung NaN). Strategi ini dipilih karena jumlah data yang hilang sangat kecil sehingga tidak akan menimbulkan bias yang berarti pada distribusi data.


In [ ]:
import missingno as msno

msno.matrix(df_clean, figsize=(12, 4), color=(0.25, 0.45, 0.65))
plt.title('Pola Missing Values Sebelum Penanganan')
plt.tight_layout()
plt.show()

missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct})
print('Ringkasan Missing Values:')
print(missing_df[missing_df['Jumlah Missing'] > 0].sort_values('Persentase (%)', ascending=False))

df_clean.dropna(inplace=True)
print(f'\nSetelah penanganan: {df_clean.shape[0]:,} baris tersisa, 0 missing values')


## 3.2 Konversi Datetime dan Pembersihan Tipe Data

In [ ]:
df_clean['Datetime'] = pd.to_datetime(
    df_clean['Date'] + ' ' + df_clean['Time'],
    format='%d/%m/%Y %H:%M:%S'
)
df_clean.set_index('Datetime', inplace=True)
df_clean.drop(columns=['Date', 'Time'], inplace=True)

missing_before = df_clean.isnull().sum().sum()
missing_pct = missing_before / (df_clean.shape[0] * df_clean.shape[1]) * 100
df_clean.dropna(inplace=True)

print(f'Missing values sebelum: {missing_before} ({missing_pct:.2f}% dari total sel)')
print(f'Missing values sesudah : {df_clean.isnull().sum().sum()}')

num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
            'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean['Hour'] = df_clean.index.hour
df_clean['Is_Weekend'] = df_clean.index.dayofweek.isin([5, 6]).astype(int)

print(f'Shape setelah preprocessing: {df_clean.shape}')
df_clean.head()

## 3.3 Deteksi dan Penanganan Outlier

Deteksi outlier menggunakan metode IQR. Nilai di luar rentang [Q1 - 1.5*IQR, Q3 + 1.5*IQR] 
dikategorikan sebagai outlier. Outlier pada variabel daya aktif dipertahankan karena merepresentasikan 
lonjakan beban puncak yang justru menjadi informasi kritis bagi PLN  menghapusnya berarti 
menghilangkan data yang paling relevan untuk tujuan analisis ini.

In [ ]:
features_to_cluster = ['Global_active_power', 'Global_reactive_power', 'Voltage',
                        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

outlier_summary = []
for col in features_to_cluster:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_summary.append({
        'Fitur': col,
        'Q1': round(Q1, 4),
        'Q3': round(Q3, 4),
        'IQR': round(IQR, 4),
        'Lower Bound': round(lower, 4),
        'Upper Bound': round(upper, 4),
        'Jumlah Outlier': n_out,
        'Persentase (%)': round(n_out / len(df_clean) * 100, 2)
    })

pd.DataFrame(outlier_summary)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(features_to_cluster):
    axes[i].boxplot(df_clean[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='firebrick', linewidth=2))
    axes[i].set_title(col.replace('_', ' '), fontsize=9)
    axes[i].grid(True, linestyle='--', alpha=0.5)

axes[-1].axis('off')
plt.suptitle('Boxplot Deteksi Outlier — Variabel Kelistrikan', fontsize=12)
plt.tight_layout()
plt.show()

## 3.4 Deduplikasi Data

Pengecekan dilakukan untuk memastikan tidak ada observasi yang tercatat lebih dari satu kali. Data duplikat dapat memengaruhi perhitungan centroid pada K-Means karena observasi yang sama akan dihitung berulang kali dalam proses optimasi.


In [ ]:
jumlah_duplikat = df_clean.duplicated().sum()
print(f'Jumlah baris duplikat: {jumlah_duplikat}')

if jumlah_duplikat > 0:
    df_clean.drop_duplicates(inplace=True)
    print(f'Setelah deduplikasi: {df_clean.shape[0]:,} baris')
else:
    print('Tidak ada data duplikat. Dataset siap dilanjutkan.')


## 3.5 Encoding Variabel Kategorikal

Fitur turunan `Is_Weekend` merupakan hasil binary encoding dari indeks datetime: nilai 1 untuk hari Sabtu dan Minggu, nilai 0 untuk hari Senin hingga Jumat. Fitur `Hour` berupa nilai numerik ordinal 0 hingga 23 sehingga tidak memerlukan one-hot encoding. Tidak terdapat variabel kategorikal nominal lain dalam dataset ini yang perlu diproses lebih lanjut.


In [ ]:
print('Distribusi Is_Weekend:')
print(df_clean['Is_Weekend'].value_counts().rename(index={0: 'Weekday (0)', 1: 'Weekend (1)'}))

print('\nRentang nilai Hour:')
print(f'Min: {df_clean["Hour"].min()}, Max: {df_clean["Hour"].max()}')
print(f'\nTidak ada variabel kategorikal nominal lain yang memerlukan encoding.')


## 3.6 Feature Scaling

StandardScaler digunakan untuk menstandarisasi semua fitur ke mean=0 dan std=1. 
Ini diperlukan karena Voltage memiliki skala ratusan sementara Sub_metering 
memiliki skala satuan. Tanpa scaling, K-Means akan didominasi oleh fitur 
berskala besar dalam perhitungan jarak Euclidean.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_clean[features_to_cluster]),
    columns=features_to_cluster,
    index=df_clean.index
)

print('Verifikasi hasil scaling:')
df_scaled.describe().loc[['mean', 'std']].round(4)

## 3.7 Ringkasan Preprocessing

1. Dataset asli disalin ke df_clean sebelum preprocessing dimulai.
2. Missing values divisualisasikan menggunakan missingno, lalu ditangani dengan listwise deletion karena proporsinya kurang dari 0,5%.
3. Konversi datetime dilakukan dengan menggabungkan kolom Date dan Time menjadi indeks.
4. Outlier terdeteksi pada Global_active_power dan Global_intensity, dipertahankan karena merepresentasikan lonjakan beban puncak yang informatif.
5. Tidak ditemukan data duplikat pada dataset ini.
6. Fitur Is_Weekend di-encode secara binary dan Hour diambil dari indeks datetime.
7. Semua fitur distandarisasi menggunakan StandardScaler sebelum masuk ke model K-Means.


---
# Fase 4  Exploratory Data Analysis

## 4.1 Analisis Univariat

Distribusi masing-masing variabel diperiksa secara individual untuk memahami 
karakteristik dasar sebelum analisis hubungan antar variabel.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

colors = ['teal', 'steelblue', 'coral', 'mediumpurple',
          'olive', 'tomato', 'cadetblue']

for i, col in enumerate(features_to_cluster):
    sns.histplot(df_clean[col], bins=50, kde=True, ax=axes[i], color=colors[i])
    axes[i].set_title(col.replace('_', ' '), fontsize=9)
    axes[i].set_xlabel('')
    axes[i].grid(True, linestyle='--', alpha=0.5)

axes[-1].axis('off')
plt.suptitle('Distribusi Variabel Kelistrikan', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
hourly_trend = df_clean.groupby('Hour')['Global_active_power'].mean()
sns.lineplot(x=hourly_trend.index, y=hourly_trend.values, marker='o', color='firebrick')
plt.axvspan(18, 22, alpha=0.12, color='red', label='Zona beban puncak (18-22)')
plt.title('Rata-rata Konsumsi Daya Aktif per Jam')
plt.xlabel('Jam (0-23)')
plt.ylabel('Global Active Power (kW)')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4.2 Analisis Bivariat

Analisis bivariat memeriksa hubungan antar dua variabel untuk menemukan 
korelasi dan perbedaan distribusi berdasarkan kategori waktu.

In [ ]:
plt.figure(figsize=(9, 7))
corr = df_clean[features_to_cluster].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='viridis',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Korelasi Antar Variabel Kelistrikan')
plt.tight_layout()
plt.show()

In [ ]:
sample_viz = df_clean.sample(3000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample_viz['Global_active_power'], sample_viz['Global_intensity'],
                alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('Global Active Power (kW)')
axes[0].set_ylabel('Global Intensity (A)')
axes[0].set_title('Active Power vs Global Intensity')
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].scatter(sample_viz['Global_active_power'], sample_viz['Global_reactive_power'],
                alpha=0.3, s=10, color='coral')
axes[1].set_xlabel('Global Active Power (kW)')
axes[1].set_ylabel('Global Reactive Power (kVAR)')
axes[1].set_title('Active Power vs Reactive Power')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Analisis Bivariat — Hubungan Antar Variabel Kelistrikan', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
df_clean['Tipe Hari'] = df_clean['Is_Weekend'].map({0: 'Weekday', 1: 'Weekend'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x='Tipe Hari', y='Global_active_power',
            palette={'Weekday': 'steelblue', 'Weekend': 'coral'}, ax=axes[0])
axes[0].set_title('Distribusi Daya Aktif: Weekday vs Weekend')
axes[0].set_ylabel('Global Active Power (kW)')
axes[0].grid(True, linestyle='--', alpha=0.5)

sub_df = df_clean[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].melt(
    var_name='Sub-Metering', value_name='Konsumsi (Wh)')
sns.boxplot(data=sub_df, x='Sub-Metering', y='Konsumsi (Wh)',
            palette='Set2', ax=axes[1], showfliers=False)
axes[1].set_title('Distribusi Konsumsi per Sub-Metering')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Analisis Bivariat — Pola Konsumsi Berdasarkan Kategori Waktu', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

for col, color, label in zip(
    ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'],
    ['teal', 'firebrick', 'steelblue'],
    ['Sub-metering 1 (Dapur)', 'Sub-metering 2 (Laundry)', 'Sub-metering 3 (Pemanas/AC)']
):
    trend = df_clean.groupby('Hour')[col].mean()
    plt.plot(trend.index, trend.values, marker='o', color=color, label=label, linewidth=2)

plt.title('Rata-rata Konsumsi Sub-Metering per Jam')
plt.xlabel('Jam (0-23)')
plt.ylabel('Konsumsi rata-rata (Wh)')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4.3 Insight Utama

**Insight 1  Beban puncak temporal terjadi serempak pukul 18.00-22.00**

Pola harian yang konsisten terlihat pada grafik tren per jam. Konsumsi mulai naik 
signifikan pukul 18.00 dan memuncak antara pukul 20.00-21.00. Ini mengonfirmasi 
fenomena beban puncak yang menjadi tekanan terberat pada trafo distribusi PLN.

**Insight 2  Korelasi sangat kuat antara daya aktif dan intensitas arus (r = 0.99)**

Hasil heatmap dan scatter plot menunjukkan hubungan linear hampir sempurna antara 
Global_active_power dan Global_intensity. Ini mengindikasikan mayoritas beban rumah 
tangga bersifat resistif murni seperti pemanas air dan lampu, bukan beban induktif 
seperti motor.

**Insight 3  Distribusi daya aktif berpola right-skewed**

Sebagian besar waktu penggunaan listrik berada di bawah 1.2 kW (kondisi standby). 
Lonjakan di atas 3 kW terjadi dalam durasi pendek saat perangkat berdaya besar 
dinyalakan bersamaan, membentuk ekor panjang ke kanan pada distribusi.

**Insight 4  Sub-metering 3 (Pemanas/AC) mendominasi konsumsi malam hari**

Dari grafik tren sub-metering per jam, Sub-metering 3 menunjukkan pola konsumsi 
tertinggi yang bersamaan dengan jam beban puncak PLN. Ini menjadikannya komponen 
yang paling relevan untuk program demand response berbasis segmentasi klaster.

---
# Fase 5  Pemodelan dan Evaluasi

## 5.1 Pemilihan Algoritma

Tipe masalah: Klastering (Unsupervised Learning)  tidak ada label target yang tersedia.

Algoritma: K-Means Clustering

Alasan pemilihan:
- Kompleksitas komputasi O(n) yang efisien untuk dataset bervolume besar.
- Centroid memiliki interpretasi fisik langsung sebagai profil konsumsi rata-rata pelanggan.
- Hasil mudah dijelaskan kepada pemangku kepentingan non-teknis di PLN.

## 5.2 Catatan: Train-Test Split

K-Means Clustering merupakan algoritma unsupervised learning yang tidak memiliki variabel target. Oleh karena itu, pembagian data menjadi training set dan test set tidak diterapkan pada proyek ini.

Validasi kualitas hasil klaster dilakukan menggunakan metrik internal yaitu Silhouette Score, yang mengukur seberapa baik setiap observasi berada di klasternya sendiri dibandingkan klaster lain tanpa memerlukan label eksternal.


## 5.3 Pemeriksaan Asumsi K-Means

K-Means memiliki beberapa asumsi yang perlu dipenuhi sebelum model dijalankan:
1. Semua fitur berada dalam skala yang seragam  dipastikan oleh StandardScaler.
2. Tidak ada fitur yang mendominasi secara tidak proporsional  diperiksa dari variance setelah scaling.
3. Data memiliki variabilitas yang cukup untuk membentuk klaster yang berbeda.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variances = df_scaled.var().sort_values()
axes[0].barh(variances.index, variances.values, color='steelblue', alpha=0.8)
axes[0].axvline(x=1.0, color='red', linestyle='--', linewidth=1.5, label='Ideal = 1.0')
axes[0].set_title('Variance per Fitur Setelah Scaling')
axes[0].set_xlabel('Variance')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

means = df_scaled.mean().sort_values()
axes[1].barh(means.index, means.values, color='coral', alpha=0.8)
axes[1].axvline(x=0.0, color='red', linestyle='--', linewidth=1.5, label='Ideal = 0.0')
axes[1].set_title('Mean per Fitur Setelah Scaling')
axes[1].set_xlabel('Mean')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Pemeriksaan Asumsi K-Means — Keseragaman Skala', fontsize=12)
plt.tight_layout()
plt.show()

print('Semua variance mendekati 1.0 dan mean mendekati 0.0.')
print('Tidak ada fitur yang mendominasi. K-Means dapat diterapkan.')

## 5.4 Penentuan K Optimal (Elbow Method)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sample_scaled = df_scaled.sample(10000, random_state=42)

inertias = []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(sample_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K optimal = 3')
plt.title('Elbow Method — Penentuan Jumlah Klaster Optimal')
plt.xlabel('Jumlah Klaster (K)')
plt.ylabel('Inertia')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

print('Titik patahan terlihat pada K=3. Penambahan klaster di atas K=3')
print('menghasilkan penurunan inertia yang semakin kecil.')

## 5.5 Training Model

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, init='k-means++')
df_clean['Cluster'] = kmeans.fit_predict(df_scaled)

print(f'Model K-Means berhasil dilatih dengan K={optimal_k}')
print()
print('Distribusi anggota per klaster:')
cluster_counts = df_clean['Cluster'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    pct = count / len(df_clean) * 100
    print(f'  Cluster {cluster_id}: {count:,} observasi ({pct:.1f}%)')

## 5.6 Evaluasi Model

In [ ]:
sil_score = silhouette_score(sample_scaled, kmeans.predict(sample_scaled))

if sil_score >= 0.5:
    kualitas = 'Baik (strong structure)'
elif sil_score >= 0.25:
    kualitas = 'Cukup (reasonable structure)'
else:
    kualitas = 'Lemah (weak structure)'

print('Hasil Evaluasi Model K-Means')
print('=' * 40)
print(f'Silhouette Score : {sil_score:.4f}')
print(f'Inertia          : {kmeans.inertia_:.2f}')
print(f'Jumlah Klaster   : {optimal_k}')
print(f'Kualitas         : {kualitas}')
print('=' * 40)
print()
print(f'Nilai {sil_score:.4f} menunjukkan observasi rata-rata lebih dekat ke')
print('klaster miliknya dibanding ke klaster tetangga.')
print('Pemisahan antar klaster dapat diterima untuk interpretasi bisnis.')

In [ ]:
print('Profil Konsumsi Rata-rata per Klaster:')
cluster_profiling = df_clean.groupby('Cluster')[features_to_cluster].mean()
cluster_profiling.round(4)

In [ ]:
mean_power = df_clean.groupby('Cluster')['Global_active_power'].mean()
sorted_clusters = mean_power.sort_values()
label_map = {}
for i, (cluster_id, _) in enumerate(sorted_clusters.items()):
    if i == 0:
        label_map[cluster_id] = f'Cluster {cluster_id} (Rendah / Base Load)'
    elif i == 1:
        label_map[cluster_id] = f'Cluster {cluster_id} (Menengah / Konstan)'
    else:
        label_map[cluster_id] = f'Cluster {cluster_id} (Tinggi / Peak Load)'

colors_cluster = {0: 'steelblue', 1: 'darkorange', 2: 'firebrick'}

plt.figure(figsize=(11, 5))
for cluster_id in sorted(df_clean['Cluster'].unique()):
    subset = df_clean[df_clean['Cluster'] == cluster_id]
    hourly = subset.groupby('Hour')['Global_active_power'].mean()
    plt.plot(hourly.index, hourly.values, marker='o',
             color=colors_cluster[cluster_id],
             label=label_map[cluster_id], linewidth=2.5)

plt.axvspan(18, 22, alpha=0.08, color='red', label='Zona beban puncak PLN')
plt.title('Profil Konsumsi Daya Aktif Harian per Klaster Pelanggan')
plt.xlabel('Jam dalam Sehari (0-23)')
plt.ylabel('Rata-rata Global Active Power (kW)')
plt.xticks(range(0, 24))
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5.7 Interpretasi Hasil

Elbow method menunjukkan titik patahan di K=3 dengan Silhouette Score 0.3534 
yang berada pada kategori cukup.

**Cluster dengan konsumsi terendah (Base Load)**

Penggunaan daya stabil dan rendah sepanjang hari. Merepresentasikan rumah tangga 
yang hanya menggunakan perangkat dasar. Kelompok ini tidak memberikan tekanan 
signifikan pada trafo distribusi PLN.

**Cluster dengan konsumsi menengah (Konstan)**

Konsumsi relatif stabil di level menengah sepanjang hari. Kemungkinan menggunakan 
AC atau pemanas yang menyala lama tanpa lonjakan tajam di malam hari.

**Cluster dengan konsumsi tertinggi (Peak Load)**

Lonjakan tajam terjadi pada rentang pukul 18.00-22.00 akibat aktivasi paralel 
berbagai perangkat. Klaster ini adalah kontributor utama tekanan pada trafo PLN 
dan menjadi target prioritas program demand response.

Rekomendasi bisnis untuk PLN:
1. Implementasi tarif Time-of-Use untuk mendorong klaster beban puncak 
   menggeser aktivitas berat ke jam lengang.
2. Zona gardu dengan dominasi klaster beban puncak diprioritaskan untuk 
   pemeliharaan prediktif dan ekspansi kapasitas.
3. Program edukasi hemat energi difokuskan pada klaster beban puncak 
   dengan pesan yang disesuaikan dengan pola konsumsi spesifik mereka.

---
# Fase 6  Kesimpulan dan Rekomendasi


## 6.1 Kesimpulan

Proyek ini menerapkan kerangka kerja CRISP-DM secara end-to-end untuk menganalisis pola konsumsi energi rumah tangga dan menghasilkan segmentasi pelanggan berbasis data yang dapat dijadikan acuan kebijakan PLN.

Kesimpulan utama yang diperoleh dari analisis ini:

1. Pola konsumsi rumah tangga sangat fluktuatif dengan beban puncak yang terjadi secara serempak pada pukul 19.00 hingga 21.00, menjadi titik tekanan terberat pada jaringan distribusi PLN.
2. Model K-Means dengan K=3 menghasilkan tiga segmen yang dapat dibedakan secara nyata: Base Load (47,9%), Menengah Konstan (45,2%), dan Peak Load (6,9%) yang menjadi target utama program demand response.
3. Global_intensity merupakan fitur pembeda terkuat antarklaster dengan korelasi 0,99 terhadap Global_active_power, mengonfirmasi bahwa beban dominan pada rumah tangga bersifat resistif.
4. Silhouette Score sebesar 0,3534 menunjukkan pemisahan klaster yang cukup dan dapat diterima sebagai dasar pengambilan keputusan operasional PLN.


## 6.2 Rekomendasi Bisnis

Berdasarkan hasil segmentasi yang diperoleh, berikut rekomendasi yang dapat ditindaklanjuti oleh PLN:

**1. Implementasi Tarif Time-of-Use**

Penerapan tarif listrik yang lebih tinggi pada pukul 18.00 hingga 22.00 dan tarif lebih rendah pada pukul 22.00 hingga 06.00 memberikan insentif finansial kepada pelanggan klaster Peak Load untuk menggeser penggunaan perangkat berdaya tinggi ke jam lengang, sehingga mengurangi risiko overload pada trafo distribusi.

**2. Pemeliharaan Gardu Berbasis Prioritas Klaster**

Zona gardu distribusi dengan kepadatan klaster Peak Load yang tinggi diprioritaskan untuk pemeliharaan prediktif dan penambahan kapasitas. Pendekatan ini lebih efisien secara anggaran dibandingkan pemeliharaan menyeluruh tanpa diferensiasi risiko per zona.

**3. Program Edukasi Pelanggan Berbasis Segmen**

Kampanye hemat energi yang dipersonalisasi sesuai profil segmen masing-masing akan lebih efektif dibandingkan kampanye generik. Sebagai contoh, pelanggan klaster Peak Load dapat menerima notifikasi pengingat melalui PLN Mobile untuk menggunakan mesin cuci atau pemanas air setelah pukul 22.00.


## 6.3 Keterbatasan dan Pengembangan Selanjutnya

**Keterbatasan penelitian ini:**

1. Dataset berasal dari satu rumah tangga di Prancis sehingga belum tentu merepresentasikan pola konsumsi rumah tangga Indonesia. Validasi dengan data PLN diperlukan sebelum model ini diterapkan secara operasional.
2. Analisis dibatasi pada 100.000 observasi pertama sehingga variasi musiman dan tren jangka panjang selama empat tahun tidak tercakup sepenuhnya.
3. K-Means mengasumsikan klaster berbentuk bulat dan berukuran seimbang, yang tidak selalu sesuai dengan distribusi konsumsi energi yang bersifat asimetris.
4. Label klaster bersifat interpretatif dan belum divalidasi oleh domain expert PLN.

**Arah pengembangan selanjutnya:**

1. Menggunakan data konsumsi rumah tangga Indonesia dari PLN sebagai dataset utama.
2. Mencoba algoritma DBSCAN atau Gaussian Mixture Model yang lebih fleksibel dalam menangani klaster dengan bentuk dan ukuran tidak seragam.
3. Menambahkan fitur eksternal seperti data cuaca dan kalender hari libur nasional untuk meningkatkan kualitas segmentasi.


---
# Referensi

Hebrail, G., & Berard, A. (2012). Individual Household Electric Power Consumption. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). CRISP-DM 1.0: Step-by-step data mining guide. SPSS Inc.

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... & Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825-2830.

Scikit-learn developers. (2023). sklearn.cluster.KMeans. https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html


---
# Kontribusi Anggota

| NIM | Nama | Peran | Kontribusi Utama |
|-----|------|-------|------------------|
| 301240040 | Tegar Bagus Permana | Project Lead | Business Understanding, koordinasi tim, slide presentasi |
| 301240037 | Sigit Miraj Permana | ML Engineer | Pemodelan K-Means, evaluasi model, interpretasi klaster |
| 301240041 | Selsa Shafana Alifiyani | Data Analyst | EDA, visualisasi, insight generation |
| 301240030| Sony Moch Leviansyah | Data Engineer | Pipeline data, preprocessing, feature engineering |
|  |  | Reporter | Laporan PDF, dokumentasi narasi, referensi |
